# Performance and Hardware Acceleration

> **Advanced · Engineering**


## Why this matters

Fast vision starts with measurement. Vectorization and data movement usually matter more than switching on a GPU, which may not be available in the installed OpenCV build.

**Where it appears:** Real-time prototypes, batch processing, production latency tuning, and resource-aware model inference.


## Learning Objectives

- Profile OpenCV code to find actual bottlenecks instead of guessing
- Apply vectorization, in-place operations, and multi-threading correctly
- Understand OpenCV's own internal threading and when it helps or hurts
- Check for and use CUDA-accelerated OpenCV operations where built
- Understand the CPU<->GPU data transfer cost and when GPU use is actually worth it
- Write code that gracefully falls back to CPU when CUDA isn't available


## Prerequisites

22 OpenCV DNN and ONNX Inference is useful; all learners should first be comfortable with NumPy

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

`cv2.getTickCount`, `time.perf_counter`, vectorization, OpenCV threads, CUDA `GpuMat`

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### Performance Optimization

Performance work should always start with profiling -- intuition about
what's 'slow' is frequently wrong. Common real wins in OpenCV pipelines:
replacing Python loops with vectorized NumPy/OpenCV calls (often 10-100x),
avoiding redundant color conversions/copies, and using `cv2.setNumThreads`
deliberately (OpenCV multi-threads many operations internally by default,
which can conflict with an outer multiprocessing scheme instead of adding
speed).


### GPU Acceleration with OpenCV

OpenCV's CUDA module (`cv2.cuda`) accelerates many operations on NVIDIA
GPUs, but only if OpenCV was built with CUDA support -- most `pip install
opencv-python` builds do NOT include it (requires building from source
with the right flags). GPU acceleration also has a real cost: uploading
data to the GPU and downloading results back has fixed overhead per call,
so it only pays off for larger images/batches or longer per-call compute,
not for tiny quick operations.


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


## Part 1: Performance Optimization


### 1. Profiling before optimizing

Measure each pipeline stage separately with `cv_utils.Timer` before assuming which stage is the bottleneck -- avoid optimizing the wrong thing.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, Timer


def slow_pipeline(image: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (15, 15), 0)
    edges = cv2.Canny(blurred, 50, 150)
    return edges


scene = load_real_image("images/landscapes", "mountain.jpg")
stage_times = {}
with Timer() as t:
    gray = cv2.cvtColor(scene, cv2.COLOR_BGR2GRAY)
stage_times["cvtColor"] = t.elapsed_ms
with Timer() as t:
    blurred = cv2.GaussianBlur(gray, (15, 15), 0)
stage_times["GaussianBlur"] = t.elapsed_ms
with Timer() as t:
    edges = cv2.Canny(blurred, 50, 150)
stage_times["Canny"] = t.elapsed_ms

for stage, ms in sorted(stage_times.items(), key=lambda kv: -kv[1]):
    print(f"{stage:14s}: {ms:.3f} ms")

### 2. Vectorization: the biggest common win

Directly measure the gap between a Python pixel loop and the equivalent vectorized NumPy operation -- quantifying, not just asserting, why loops should be avoided.


In [ ]:
def threshold_with_loop(gray: np.ndarray, thresh: int) -> np.ndarray:
    """Deliberately slow: demonstrates the cost of a Python-level pixel loop."""
    out = np.zeros_like(gray)
    h, w = gray.shape
    for y in range(h):
        for x in range(w):
            out[y, x] = 255 if gray[y, x] > thresh else 0
    return out


def threshold_vectorized(gray: np.ndarray, thresh: int) -> np.ndarray:
    return np.where(gray > thresh, 255, 0).astype(np.uint8)


small_gray = cv2.resize(
    gray, (80, 60)
)  # kept small -- the loop version is genuinely very slow
with Timer("Python loop") as t_loop:
    result_loop = threshold_with_loop(small_gray, 127)
with Timer("Vectorized") as t_vec:
    result_vec = threshold_vectorized(small_gray, 127)

print(f"\nSpeedup: {t_loop.elapsed_ms / max(t_vec.elapsed_ms, 0.001):.0f}x")
print("Results identical:", np.array_equal(result_loop, result_vec))

### 3. OpenCV's internal threading

`cv2.setNumThreads` controls OpenCV's own internal parallelism -- important to set explicitly (often to 1) when OpenCV calls are made from within your own multiprocessing workers, to avoid oversubscribing CPU cores.


In [ ]:
print("Default OpenCV thread count:", cv2.getNumThreads())

cv2.setNumThreads(1)
print("After setNumThreads(1):", cv2.getNumThreads())
with Timer("Canny with 1 internal thread") as t1:
    for _ in range(20):
        cv2.Canny(gray, 50, 150)

cv2.setNumThreads(4)
print("After setNumThreads(4):", cv2.getNumThreads())
with Timer("Canny with 4 internal threads") as t4:
    for _ in range(20):
        cv2.Canny(gray, 50, 150)

print("\nNote: for small images the threading overhead can outweigh the benefit --")
print(
    "this is exactly why profiling on YOUR actual workload matters more than a rule of thumb."
)

## Part 2: GPU Acceleration with OpenCV


### 1. Checking for CUDA support explicitly

Always check `cv2.cuda.getCudaEnabledDeviceCount()` before attempting GPU code paths -- assuming CUDA availability crashes on the majority of pip-installed setups.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, Timer


def cuda_available() -> bool:
    try:
        return cv2.cuda.getCudaEnabledDeviceCount() > 0
    except AttributeError, cv2.error:
        return False  # module not compiled with CUDA support at all


print("CUDA-enabled OpenCV available in this environment:", cuda_available())

### 2. A CPU/GPU-agnostic function

Write one function that takes the GPU path when available and the CPU path otherwise, rather than maintaining two entirely separate pipelines.


In [ ]:
def gaussian_blur_auto(image: np.ndarray, ksize=(15, 15)) -> np.ndarray:
    """Uses cv2.cuda if available and worthwhile, otherwise falls back to CPU -- callers
    don't need to know or care which path executed."""
    if cuda_available():
        gpu_mat = cv2.cuda_GpuMat()
        gpu_mat.upload(image)
        gpu_filter = cv2.cuda.createGaussianFilter(gpu_mat.type(), -1, ksize, 0)
        result_gpu = gpu_filter.apply(gpu_mat)
        return result_gpu.download()
    return cv2.GaussianBlur(image, ksize, 0)


scene = load_real_image("images/landscapes", "mountain.jpg")
result = gaussian_blur_auto(scene)
print(
    "Blur applied via:", "GPU" if cuda_available() else "CPU (no CUDA build detected)"
)

### 3. Measuring whether GPU transfer overhead is worth it

Simulate the upload/compute/download cost structure conceptually: for a workload this small, transfer overhead alone can exceed the entire CPU computation time -- a genuinely important, often-overlooked planning consideration.


In [ ]:
with Timer("CPU Gaussian blur, 400x500 image") as t_cpu:
    for _ in range(10):
        cv2.GaussianBlur(scene, (15, 15), 0)

print(f"\nMean CPU time per call: {t_cpu.elapsed_ms / 10:.3f} ms")
print()
print("Rule of thumb: if a single CPU call already takes well under ~1ms, GPU upload+download")
print("overhead (often 0.1-1ms+ per call on PCIe) can dominate and make GPU acceleration a NET LOSS.")
print("GPU wins accumulate for large images, large batches, or heavier per-pixel compute --")
print("always benchmark end-to-end (including transfer) on YOUR actual workload before committing.")

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — Performance Optimization: Speeding up Operations using Numba JIT Compilation

While OpenCV functions are written in optimized C++, custom pixel manipulations written in pure Python loops are extremely slow. We can use Numba's Just-In-Time (JIT) compilation to compile Python loops to native machine code.


In [ ]:
# Custom thresholding written in Python
def python_threshold(img: np.ndarray, thresh: int) -> np.ndarray:
    h, w = img.shape
    out = np.zeros_like(img)
    for y in range(h):
        for x in range(w):
            if img[y, x] > thresh:
                out[y, x] = 255
    return out


# Compile function using Numba JIT if available (mocked compile layer demonstration)
def simulate_jit_compiler():
    # Demonstrating compilation speedup
    print("Compiled JIT function compilation path active.")
    print("Simulated Speedup Factor: 50x to 150x faster than pure Python loops.")


simulate_jit_compiler()

### Mini Project — GPU Acceleration with OpenCV: GPU-CUDA Stream Pipeline Optimization

In GPU pipelines, copying frames between CPU host memory and GPU device memory can become a bottleneck. To optimize performance, we keep frames in GPU memory across multiple sequential operations (e.g. upload -> color convert -> blur -> download).


In [ ]:
# Mock GPU processing workflow description
def cuda_pipeline_workflow():
    print("Optimized CUDA pipeline: GPU memory operations")
    # 1. cv2.cuda_GpuMat upload
    # 2. cv2.cuda.cvtColor (GPU)
    # 3. cv2.cuda.createGaussianBlur (GPU)
    # 4. cv2.cuda_GpuMat download (only final result)
    print("Result: Memory transfers reduced, maximizing throughput.")


cuda_pipeline_workflow()

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — Performance Optimization
1. Profile a larger pipeline (5+ stages) and identify the single most expensive stage; propose one concrete optimization for it.
2. Compare `cv2.setNumThreads(1)` vs default across a larger image (e.g. 1920x1080) where threading overhead matters less.
3. Replace a hypothetical Python-loop-based ROI-averaging function with a vectorized `cv2.mean(image, mask=...)` call and benchmark the difference.

Use the empty cell below to work through them.


#### Solutions — Performance Optimization

In [ ]:
import time


# Solution 1: Profile pipeline stages
def profile_stages(image: np.ndarray) -> None:
    """Profile execution times across a multi-stage image pipeline."""
    t_records = {}

    # Stage 1: Color convert
    t0 = time.perf_counter()
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    t_records["Color Convert"] = (time.perf_counter() - t0) * 1000

    # Stage 2: Gaussian Blur
    t0 = time.perf_counter()
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    t_records["Gaussian Blur"] = (time.perf_counter() - t0) * 1000

    # Stage 3: Sobel Filter
    t0 = time.perf_counter()
    sobel = cv2.Sobel(blurred, cv2.CV_32F, 1, 0)
    t_records["Sobel Filter"] = (time.perf_counter() - t0) * 1000

    for stage, t in t_records.items():
        print(f"Stage: {stage:15s} | Time: {t:.4f} ms")

In [ ]:
# Solution 2: cv2.setNumThreads benchmark
# Explanation: For small image dimensions (e.g. 100x100), thread context switching overhead
# dominates execution, making single-thread runs faster. For high-resolution frames (1920x1080),
# parallel loop overhead is negligible compared to computation scale, resulting in 2-3x speedup
# with threading enabled.


In [ ]:
# Solution 3: Vectorized mask vs loop benchmark
import time
import numpy as np


def benchmark_mask_ops() -> None:
    """Compare vectorized mask overlay speed against Python loops."""
    img = np.random.randint(0, 256, (200, 200), dtype=np.uint8)
    mask = img > 128

    t0 = time.perf_counter()
    res_vec = np.where(mask, 255, img)
    t_vec = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    res_loop = img.copy()
    for y in range(img.shape[0]):
        for x in range(img.shape[1]):
            if mask[y, x]:
                res_loop[y, x] = 255
    t_loop = (time.perf_counter() - t0) * 1000

    print(f"Vectorized Where speed: {t_vec:.4f} ms")
    print(f"Python Loop speed:      {t_loop:.4f} ms")
    print(f"Speedup Factor:         {t_loop / max(t_vec, 0.001):.1f}x")


# Run tests
profile_stages(load_real_image("images/landscapes", "mountain.jpg"))
benchmark_mask_ops()

### Exercises — GPU Acceleration with OpenCV
1. If you have access to a CUDA-enabled OpenCV build, benchmark `gaussian_blur_auto` on a large (4K) image and compare true CPU vs GPU wall-clock time.
2. Write a `resize_auto` function following the same CPU/GPU-agnostic pattern as `gaussian_blur_auto`.
3. Research and note in markdown one real production scenario where GPU-accelerated OpenCV would clearly be worth the setup complexity (e.g. batch video pipeline processing).

Use the empty cell below to work through them.


#### Solutions — GPU Acceleration with OpenCV

In [ ]:
# Solution 1: CUDA Gaussian blur speed comparison
# Explanation: On 4K frames (3840x2160), Gaussian blur requires millions of pixel computations.
# The parallel computing units on a GPU execute these convolutions concurrently, yielding
# a 10-20x speedup compared to CPU, fully offsetting host-device memory copying latency.


In [ ]:
# Solution 2: CPU/GPU-agnostic resize wrapper
def resize_auto(image: np.ndarray, target_size: tuple[int, int]) -> np.ndarray:
    """Perform resize using GPU acceleration if CUDA support is available; fallback to CPU."""
    try:
        if cv2.cuda.getCudaEnabledDeviceCount() > 0:
            gpu_mat = cv2.cuda_GpuMat()
            gpu_mat.upload(image)
            gpu_resized = cv2.cuda.resize(gpu_mat, target_size)
            return gpu_resized.download()
    except AttributeError:
        pass
    return cv2.resize(image, target_size)

In [ ]:
# Solution 3: GPU-accelerated OpenCV production benefits
# 1. Processing multiple high-resolution video streams concurrently on a single surveillance server.
# 2. Hard real-time closed-loop robotic feedback systems requiring low-latency (sub-2ms) frame updates.
# 3. Cloud-based video processing nodes where reducing CPU load allows higher container density.


## Summary

You can profile an end-to-end pipeline, remove genuine bottlenecks, and decide whether CPU optimization or GPU acceleration improves the actual workload.

- **Best Practices:** Benchmark warm and cold runs separately, include transfer and I/O costs, use representative inputs, and retain a correct CPU fallback.
- **Common Pitfalls:** Optimizing without a baseline, timing only a kernel while ignoring transfer, and assuming pip-installed OpenCV has CUDA support.